# **Task - 1**

In [8]:
STATES = ["S1", "S2", "S3", "S4", "S5"]
N = len(STATES)
TERMINAL = "S5"
ACTIONS = ["L", "R"]
GAMMA = 0.9
MAX_STEPS = 50

def idx(s):
    return int(s[1:]) - 1

def step(s, a):
    """Return (next_state, reward)"""
    if s == TERMINAL:
        return s, 0
    i = idx(s)
    if a == "L":
        i = max(0, i - 1)
    else:  # "R"
        i = min(N - 1, i + 1)
    ns = STATES[i]
    r = 0 if ns == TERMINAL else -1
    return ns, r


# **Task - 2**

In [2]:
given_policy = {s: {"L": 0.3, "R": 0.7} for s in STATES}

def policy_evaluation(policy, theta=1e-4, max_iterations=5, verbose_rows=None):
    V = {s: 0.0 for s in STATES}
    history = []
    for it in range(1, max_iterations + 1):
        delta = 0.0
        V_new = V.copy()
        for s in STATES:
            if s == TERMINAL:
                continue
            v_new = 0.0
            for a in ACTIONS:
                p = policy[s][a]
                ns, r = step(s, a)
                v_new += p * (r + GAMMA * V[ns])
            V_new[s] = v_new
            delta = max(delta, abs(v_new - V[s]))
        V = V_new
        history.append((it, delta))
        if verbose_rows is not None:
            verbose_rows.append((it, round(delta, 5), "Converged" if delta < theta else "Not yet"))
        if delta < theta:
            break
    return V, history

eval_rows = []
V_eval, hist = policy_evaluation(given_policy, max_iterations=5, verbose_rows=eval_rows)

print("=== Task 2: Policy Evaluation (Given Policy 70% R / 30% L) ===")
for it, delta, status in eval_rows:
    print(f"Iter {it}: max delta = {delta}  -> {status}")
print("Converged state values:", {s: round(v, 3) for s, v in V_eval.items()})
print()

=== Task 2: Policy Evaluation (Given Policy 70% R / 30% L) ===
Iter 1: max delta = 1.0  -> Not yet
Iter 2: max delta = 0.9  -> Not yet
Iter 3: max delta = 0.81  -> Not yet
Iter 4: max delta = 0.55397  -> Not yet
Iter 5: max delta = 0.45131  -> Not yet
Converged state values: {'S1': -3.715, 'S2': -3.2, 'S3': -2.293, 'S4': -0.865, 'S5': 0.0}



# **Task - 3**

In [3]:
def value_iteration(theta=1e-6, max_iterations=1000):
    V = {s: 0.0 for s in STATES}
    for it in range(1, max_iterations + 1):
        delta = 0.0
        V_new = V.copy()
        for s in STATES:
            if s == TERMINAL:
                continue
            action_values = {}
            for a in ACTIONS:
                ns, r = step(s, a)
                action_values[a] = r + GAMMA * V[ns]
            best = max(action_values.values())
            V_new[s] = best
            delta = max(delta, abs(best - V[s]))
        V = V_new
        if delta < theta:
            break

    # derive greedy optimal policy
    policy = {}
    for s in STATES:
        if s == TERMINAL:
            policy[s] = "-"
            continue
        action_values = {}
        for a in ACTIONS:
            ns, r = step(s, a)
            action_values[a] = r + GAMMA * V[ns]
        policy[s] = max(action_values, key=action_values.get)
    return V, policy, it

V_star, optimal_policy, n_iters = value_iteration()
print("=== Task 3: Value Iteration ===")
print(f"Converged after {n_iters} iterations")
for s in STATES:
    print(f"{s}: optimal action = {optimal_policy[s]:>1}  V*(s) = {round(V_star[s], 3)}")
print()

=== Task 3: Value Iteration ===
Converged after 4 iterations
S1: optimal action = R  V*(s) = -2.71
S2: optimal action = R  V*(s) = -1.9
S3: optimal action = R  V*(s) = -1.0
S4: optimal action = R  V*(s) = 0.0
S5: optimal action = -  V*(s) = 0.0



# **Task - 4**

In [5]:
import random

random.seed(42)

def run_random_policy(start="S1", max_steps=MAX_STEPS):
    s = start
    path = [s]
    total_reward = 0
    for _ in range(max_steps):
        if s == TERMINAL:
            break
        a = random.choice(ACTIONS)
        ns, r = step(s, a)
        total_reward += r
        path.append(ns)
        s = ns
    reached = path[-1] == TERMINAL
    return path, len(path) - 1, total_reward, reached

def run_stochastic_policy(policy, start="S1", max_steps=MAX_STEPS):
    s = start
    path = [s]
    total_reward = 0
    for _ in range(max_steps):
        if s == TERMINAL:
            break
        a = random.choices(ACTIONS, weights=[policy[s]["L"], policy[s]["R"]])[0]
        ns, r = step(s, a)
        total_reward += r
        path.append(ns)
        s = ns
    reached = path[-1] == TERMINAL
    return path, len(path) - 1, total_reward, reached

def run_deterministic_policy(policy, start="S1", max_steps=MAX_STEPS):
    s = start
    path = [s]
    total_reward = 0
    for _ in range(max_steps):
        if s == TERMINAL:
            break
        a = policy[s]
        ns, r = step(s, a)
        total_reward += r
        path.append(ns)
        s = ns
    reached = path[-1] == TERMINAL
    return path, len(path) - 1, total_reward, reached

random_path, random_len, random_reward, random_reached = run_random_policy()
eval_path, eval_len, eval_reward, eval_reached = run_stochastic_policy(given_policy)
opt_path, opt_len, opt_reward, opt_reached = run_deterministic_policy(optimal_policy)

print("=== Task 4: Path Comparison (start = S1) ===")
print("Random Policy   :", "->".join(random_path), "| steps:", random_len, "| reward:", random_reward, "| goal reached:", random_reached)
print("Evaluated Policy:", "->".join(eval_path), "| steps:", eval_len, "| reward:", eval_reward, "| goal reached:", eval_reached)
print("Optimal Policy  :", "->".join(opt_path), "| steps:", opt_len, "| reward:", opt_reward, "| goal reached:", opt_reached)


=== Task 4: Path Comparison (start = S1) ===
Random Policy   : S1->S1->S1->S2->S1->S1->S1->S1->S1->S2->S1->S1->S1->S1->S1->S1->S1->S2->S1->S2->S3->S2->S1->S2->S3->S4->S3->S2->S3->S2->S1->S2->S1->S2->S3->S4->S3->S4->S3->S4->S3->S4->S5 | steps: 42 | reward: -41 | goal reached: True
Evaluated Policy: S1->S2->S3->S2->S1->S1->S1->S1->S1->S1->S2->S3->S4->S3->S2->S3->S4->S5 | steps: 17 | reward: -16 | goal reached: True
Optimal Policy  : S1->S2->S3->S4->S5 | steps: 4 | reward: -3 | goal reached: True
